# 양자컴퓨팅 부트캠프 Day 1: 선형대수학과 단일 큐비트 상태벡터 NumPy 핸즈온

본 노트북은 **12시간 양자컴퓨팅 특강의 1일차 3교시 실습 교재**입니다.
복소 벡터 공간, 유니터리 연산자, Born 측정 규칙을 순수 Python과 `numpy`만으로 직접 계산하고 시뮬레이션합니다.

---

### 오늘 실습의 5대 학습 목표
1. **복소수와 계산 기저**: 복소 내적 공간 $\mathbb{C}^2$와 계산 기저 $|0\rangle, |1\rangle$를 2차원 열벡터로 정의.
2. **단일 큐비트 유니터리 게이트**: $I, X, Y, Z, H, S, T$ 게이트의 행렬 정의와 유니터리성($U^* U = I$) 전수 검증.
3. **정규화와 Born 측정 규칙**: 상태벡터의 크기 보존($\langle\psi|\psi\rangle = 1$)과 측정 확률 $P(0)=|\alpha|^2, P(1)=|\beta|^2$ 계산.
4. **Hadamard 기저 변환과 가역성**: 계산 기저($|0\rangle, |1\rangle$)와 중첩 기저($|+\rangle, |-\rangle$)의 상호 1:1 변환 및 자기 역원성($H^2 = I$) 검증.
5. **Shot Noise와 표본 수렴**: 유한한 반복 측정(Shots)에 따른 통계적 오차가 대수의 법칙($1/\sqrt{M}$)에 따라 수렴함을 확인.


In [ ]:
from __future__ import annotations

import numpy as np

# 소수점 출력 포맷 설정
np.set_printoptions(precision=4, suppress=True)

print(f'NumPy version: {np.__version__}')


## 1. 계산 기저 (Computational Basis) 상태 정의

단일 큐비트의 물리적 상태는 2차원 복소 내적 공간 $\mathbb{C}^2$의 단위 열벡터로 기술됩니다.
가장 기본이 되는 표준 정규직교기저 $|0\rangle$과 $|1\rangle$을 정의합니다:

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$


In [ ]:
# 2차원 복소 열벡터 정의
KET_0 = np.array([[1.0 + 0.0j], [0.0 + 0.0j]])  # |0>
KET_1 = np.array([[0.0 + 0.0j], [1.0 + 0.0j]])  # |1>

print('|0> basis vector:')
print(KET_0)
print('\n|1> basis vector:')
print(KET_1)


## 2. 단일 큐비트 유니터리 게이트 (Unitary Gates)

양자 게이트는 닫힌 양자계의 가역적인 물리적 시간 발전을 나타내며, 선형대수학적으로 **크기를 보존하는 $2 \times 2$ 유니터리 행렬**로 표현됩니다:

$$U^* U = U U^* = I \iff U^{-1} = U^*$$

주요 단일 큐비트 게이트를 정의합니다:
- **Pauli-X (비트 반전, NOT)**: $X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$
- **Pauli-Y**: $Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$
- **Pauli-Z (위상 반전)**: $Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$
- **Hadamard (균등 중첩 생성)**: $H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$
- **Phase ($S = Z^{1/2}$)**: $S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}$
- **T ($T = Z^{1/4}$)**: $T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix}$


In [ ]:
GATE_I = np.array([[1, 0], [0, 1]], dtype=complex)
GATE_X = np.array([[0, 1], [1, 0]], dtype=complex)
GATE_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
GATE_Z = np.array([[1, 0], [0, -1]], dtype=complex)
GATE_H = (1.0 / np.sqrt(2.0)) * np.array([[1, 1], [1, -1]], dtype=complex)
GATE_S = np.array([[1, 0], [0, 1j]], dtype=complex)
GATE_T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)

def is_unitary(matrix: np.ndarray, tol: float = 1e-9) -> bool:
    dim = matrix.shape[0]
    identity = np.eye(dim, dtype=complex)
    product = matrix.conj().T @ matrix
    return bool(np.allclose(product, identity, atol=tol))

gates = {
    'I': GATE_I, 'X': GATE_X, 'Y': GATE_Y, 'Z': GATE_Z,
    'H': GATE_H, 'S': GATE_S, 'T': GATE_T
}

print('[게이트 유니터리성 검증]')
for name, g in gates.items():
    valid = is_unitary(g)
    print(f'  - Gate {name:2s}: Unitary = {valid}')
    assert valid, f'{name} 게이트는 유니터리하지 않습니다!'
print('=> 모든 기본 게이트가 벡터 크기(노름)를 보존함을 확인했습니다.')


## 3. 정규화와 Born 측정 규칙

상태벡터 $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$를 측정하면:
1. $0$이 관측될 확률: $P(0) = |\alpha|^2$
2. $1$이 관측될 확률: $P(1) = |\beta|^2$
3. 전체 확률 합: $|\alpha|^2 + |\beta|^2 = \langle\psi|\psi\rangle = 1$ (정규화 조건)


In [ ]:
def check_normalization(state: np.ndarray, tol: float = 1e-9) -> bool:
    norm_sq = np.vdot(state, state).real
    return bool(np.isclose(norm_sq, 1.0, atol=tol))

def measure_probabilities(state: np.ndarray) -> tuple[float, float]:
    p0 = float(np.abs(state[0, 0]) ** 2)
    p1 = float(np.abs(state[1, 0]) ** 2)
    return p0, p1

sample_state = np.array([[0.5], [np.sqrt(3)/2.0]], dtype=complex)

print('상태벡터 정규화 검증:', check_normalization(sample_state))
p0, p1 = measure_probabilities(sample_state)
print(f'이론적 확률: P(0) = {p0:.4f} (25%), P(1) = {p1:.4f} (75%)')


## 4. Hadamard 기저 변환과 자기 역원성 ($H^2 = I$)

Hadamard 게이트 $H$는 계산 기저($Z$축: $\{|0\rangle, |1\rangle\}$)와 중첩 기저($X$축: $\{|+\rangle, |-\rangle\}$)를 상호 변환하는 가역적 연산자입니다:

1. **순방향 변환 ($Z \to X$)**:
   $$H|0\rangle = |+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \qquad H|1\rangle = |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$
2. **역방향 변환 ($X \to Z$)**:
   $$H|+\rangle = |0\rangle, \qquad H|-\rangle = |1\rangle$$
3. **자기 역원성 ($H^2 = I$)**:
   $$H \cdot H = I \implies H = H^{-1}$$

고전의 무작위 난수 생성(동전 던지기)과 달리, Hadamard 연산은 두 번 연속 적용하면 원래의 순수 상태로 완전히 복원되는 결정론적 유니터리 변환입니다.


In [ ]:
# 1. Z-기저에서 X-기저(|+>, |->)로의 순방향 변환
plus_state = GATE_H @ KET_0
minus_state = GATE_H @ KET_1

ket_plus_exact = (1.0 / np.sqrt(2.0)) * np.array([[1.0], [1.0]], dtype=complex)
ket_minus_exact = (1.0 / np.sqrt(2.0)) * np.array([[1.0], [-1.0]], dtype=complex)

print("[1. Z-기저 -> X-기저 순방향 변환]")
print("  - H @ |0> == |+>:", np.allclose(plus_state, ket_plus_exact))
print("  - H @ |1> == |->:", np.allclose(minus_state, ket_minus_exact))

p0_plus, p1_plus = measure_probabilities(plus_state)
p0_minus, p1_minus = measure_probabilities(minus_state)
print(f"  - |+> 측정 확률: P(0)={p0_plus:.4f}, P(1)={p1_plus:.4f}")
print(f"  - |-> 측정 확률: P(0)={p0_minus:.4f}, P(1)={p1_minus:.4f}")

# 2. X-기저에서 Z-기저로의 역방향 변환 (H @ |+> = |0>, H @ |-> = |1>)
recovered_0 = GATE_H @ plus_state
recovered_1 = GATE_H @ minus_state

print("\n[2. X-기저 -> Z-기저 역방향 변환]")
print("  - H @ |+> == |0> 복원 여부:", np.allclose(recovered_0, KET_0))
print("  - H @ |-> == |1> 복원 여부:", np.allclose(recovered_1, KET_1))

# 3. 자기 역원성(Involution: H^2 = I) 대수적 검증
h_squared = GATE_H @ GATE_H
print("\n[3. 자기 역원성(H^2 = I) 검증]")
print("  - H @ H == I 여부:", np.allclose(h_squared, GATE_I))

# 자가 검증 (assert)
assert np.allclose(plus_state, ket_plus_exact), "H @ |0> must be |+>"
assert np.allclose(minus_state, ket_minus_exact), "H @ |1> must be |->"
assert np.allclose(recovered_0, KET_0), "H @ |+> must recover |0>"
assert np.allclose(recovered_1, KET_1), "H @ |-> must recover |1>"
assert np.allclose(h_squared, GATE_I), "H^2 must equal I"
print("\n=> Hadamard 게이트의 1:1 기저 변환과 유니터리 가역성(H^2=I)이 모두 검증되었습니다!")


## 5. 유한한 측정과 통계적 수렴 (Shot Noise 분석)

실제 양자 컴퓨터에서는 내부의 복소 진폭을 직접 들여다볼 수 없으며, 오직 $M$번의 유한한 반복 측정(Shots)을 통해서만 고전 비트 빈도를 얻을 수 있습니다.
대수의 법칙에 따라, 관측 빈도와 이론적 확률의 오차는 $1/\sqrt{M}$에 비례하여 줄어듭니다.


In [ ]:
def sample_measurements(state: np.ndarray, shots: int = 1000) -> dict[str, int]:
    p0, p1 = measure_probabilities(state)
    outcomes = np.random.choice(['0', '1'], size=shots, p=[p0, p1])
    count_0 = int(np.sum(outcomes == '0'))
    return {'0': count_0, '1': shots - count_0}

print('[Shot 수에 따른 관측 비율 및 오차 수렴]')
true_p0 = 0.5
np.random.seed(42)

for shots in [10, 100, 1000, 10000, 100000]:
    counts = sample_measurements(plus_state, shots=shots)
    emp_p0 = counts['0'] / shots
    error = abs(emp_p0 - true_p0)
    print(f'  - Shots {shots:7d}: counts={counts} -> 관측 비율={emp_p0:.5f} (오차={error:.5f})')


---
### 6. Day 1 실습 마무리 & 다음 시간 예고

오늘 실습에서는 외부 양자 프레임워크에 의존하지 않고, **순수 Python과 NumPy**만으로 양자역학의 핵심 원리를 직접 코드로 구축하고 검증했습니다:
- 2차원 복소 단위벡터로 기술되는 단일 큐비트 상태벡터 ($|0\rangle, |1\rangle, |+\rangle, |-\rangle$)
- 상태벡터의 크기를 보존하는 유니터리 행렬곱(`@`)을 통한 양자 게이트 연산 ($U^* U = I$)
- 사영 측정과 Born 규칙 ($P(x) = |\alpha_x|^2$)
- Hadamard 게이트를 통한 기저 변환($|0\rangle \leftrightarrow |+\rangle, |1\rangle \leftrightarrow |-\rangle$)과 자기 역원성($H^2 = I$)
- 유한한 측정 횟수(Shots)와 대수의 법칙에 따른 통계적 수렴 ($1/\sqrt{M}$)

**Next Step (Day 2)**:
다음 시간에는 2차원을 넘어 **텐서곱(Tensor Product / Kronecker Product)**을 통해 다중 큐비트 공간으로 확장하고, 양자컴퓨팅의 가장 핵심 자원인 **양자 얽힘(Entanglement, Bell State)**과 2큐비트 CNOT 게이트를 직접 구현해 봅니다.
